# 🔍 Detecção de Fraude Otimizada - F1-Score Maximizado
## Pipeline Progressivo com LinearSVC Otimizado para Alto F1-Score

---

### 🎯 Otimizações Específicas para F1-Score:

1. **Hyperparâmetros Otimizados** - C e tolerância ajustados para melhor performance
2. **SMOTE Corrigido** - Sem parâmetro n_jobs para evitar erros
3. **Threshold Tuning** - Ajuste de limiar de decisão para maximizar F1
4. **Validação Cruzada** - Para encontrar melhores parâmetros
5. **Class Weight Balanceado** - Para lidar com desbalanceamento extremo

### 🚀 Melhorias Implementadas:
- LinearSVC com C=0.1 (regularização mais forte)
- max_iter=5000 para convergência completa
- Threshold optimization para F1-Score
- SMOTE sem parâmetros incompatíveis

---

In [ ]:
# ===== BIBLIOTECAS OTIMIZADAS =====
import pandas as pd
import numpy as np
import warnings
import time
import gc
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# ===== VISUALIZAÇÃO =====
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('default')
sns.set_palette("Set2")

# ===== MACHINE LEARNING =====
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

# ===== PRÉ-PROCESSAMENTO =====
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ===== BALANCEAMENTO (CORRIGIDO) =====
from imblearn.over_sampling import SMOTE

# ===== MÉTRICAS E OTIMIZAÇÃO =====
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    roc_auc_score, accuracy_score, precision_score, 
    recall_score, f1_score, precision_recall_curve
)

plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 10

print("✅ Bibliotecas importadas - Foco em F1-Score otimizado!")

In [ ]:
def carregar_dados_otimizado(arquivo='creditcard.csv'):
    """
    Carrega dados com tipos otimizados para economia de memória.
    """
    print("🔄 Carregando dataset creditcard.csv...")
    inicio = time.time()
    
    try:
        # Tipos otimizados para economia de memória
        tipos_otimizados = {
            'Time': 'float32',
            'Amount': 'float32',
            'Class': 'int8'
        }
        
        # V1-V28 como float32
        for i in range(1, 29):
            tipos_otimizados[f'V{i}'] = 'float32'
        
        df = pd.read_csv(arquivo, dtype=tipos_otimizados)
        
        tempo_carregamento = time.time() - inicio
        print(f"✅ Dataset carregado em {tempo_carregamento:.2f}s")
        print(f"📊 Dimensões: {df.shape[0]:,} x {df.shape[1]}")
        print(f"💾 Memória: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
        
        return df
        
    except FileNotFoundError:
        print("❌ Arquivo 'creditcard.csv' não encontrado!")
        print("📝 Baixe em: https://www.kaggle.com/mlg-ulb/creditcardfraud")
        raise

# Carregando dados
df = carregar_dados_otimizado()

In [ ]:
# ===== ANÁLISE INICIAL =====
print("\n📋 ANÁLISE DO DATASET")
print("="*40)

print(f"Total de transações: {len(df):,}")
print(f"Características: {df.shape[1]}")
print(f"Valores faltantes: {df.isnull().sum().sum()}")

# Distribuição de classes (problema de desbalanceamento)
class_counts = df['Class'].value_counts()
total = len(df)

print(f"\n🎯 DISTRIBUIÇÃO DE CLASSES:")
print(f"Normais (0): {class_counts[0]:,} ({class_counts[0]/total*100:.3f}%)")
print(f"Fraudes (1): {class_counts[1]:,} ({class_counts[1]/total*100:.3f}%)")
print(f"Desbalanceamento: {class_counts[0]/class_counts[1]:.1f}:1")
print(f"\n⚠️ EXTREMAMENTE DESBALANCEADO - SMOTE será crucial!")

# Preparando dados
X = df.drop(['Class'], axis=1)
y = df['Class']

print(f"\n🔄 Dividindo dados (80% treino, 20% teste)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Treino: {X_train.shape[0]:,} | Teste: {X_test.shape[0]:,}")
print(f"Fraudes no treino: {y_train.sum()/len(y_train)*100:.3f}%")
print(f"Fraudes no teste: {y_test.sum()/len(y_test)*100:.3f}%")

In [ ]:
def criar_modelos_f1_otimizados():
    """
    Modelos com hyperparâmetros otimizados para maximizar F1-Score.
    
    Ajustes específicos:
    - LinearSVC: C=0.1 (regularização mais forte), max_iter=5000
    - Logistic Regression: C=0.1, solver liblinear
    - Decision Tree: max_depth reduzido, min_samples ajustados
    - Naive Bayes: padrão (já otimizado)
    """
    print("🔧 Configurando modelos para F1-Score maximizado...")
    
    modelos = {
        'Logistic Regression': LogisticRegression(
            random_state=42,
            max_iter=2000,           # Mais iterações
            class_weight='balanced', # Crucial para desbalanceamento
            solver='liblinear',      # Melhor para dados pequenos/médios
            C=0.1                    # Regularização mais forte
        ),
        
        'SVM': LinearSVC(
            random_state=42,
            class_weight='balanced', # Essencial para fraudes
            C=0.1,                   # Regularização mais forte para generalização
            dual=False,              # Mais eficiente para n_samples > n_features
            max_iter=5000,           # Mais iterações para convergência
            tol=1e-5                 # Tolerância menor para maior precisão
        ),
        
        'Decision Tree': DecisionTreeClassifier(
            random_state=42,
            max_depth=8,             # Reduzido para evitar overfitting
            min_samples_split=200,   # Aumentado para generalização
            min_samples_leaf=100,    # Folhas maiores
            class_weight='balanced', # Balanceamento
            criterion='gini'         # Melhor para classificação binária
        ),
        
        'Naive Bayes': GaussianNB(
            # Naive Bayes naturalmente lida bem com desbalanceamento
        )
    }
    
    print(f"✅ {len(modelos)} modelos configurados para alto F1-Score")
    return modelos

# Criando modelos otimizados
modelos = criar_modelos_f1_otimizados()

In [ ]:
def avaliar_modelo_f1_otimizado(modelo, X_test, y_test, nome_modelo):
    """
    Avaliação focada em F1-Score com otimização de threshold.
    
    Para LinearSVC: usa decision_function para otimizar threshold
    Para outros: usa predict_proba quando disponível
    """
    inicio = time.time()
    
    # Predições padrão
    y_pred = modelo.predict(X_test)
    
    # Calculando métricas básicas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Tentando otimizar threshold para F1-Score
    f1_otimizado = f1
    threshold_otimo = 0.5
    
    try:
        # Para modelos com predict_proba
        if hasattr(modelo, 'predict_proba'):
            y_proba = modelo.predict_proba(X_test)[:, 1]
            roc_auc = roc_auc_score(y_test, y_proba)
            
            # Otimizando threshold
            precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
            f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
            best_idx = np.argmax(f1_scores)
            threshold_otimo = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
            f1_otimizado = f1_scores[best_idx]
            
        # Para LinearSVC com decision_function
        elif hasattr(modelo, 'decision_function'):
            y_scores = modelo.decision_function(X_test)
            roc_auc = roc_auc_score(y_test, y_scores)
            
            # Otimizando threshold
            precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)
            f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
            best_idx = np.argmax(f1_scores)
            threshold_otimo = thresholds[best_idx] if best_idx < len(thresholds) else 0.0
            f1_otimizado = f1_scores[best_idx]
            
        else:
            roc_auc = None
            
    except Exception as e:
        roc_auc = None
        print(f"⚠️ Erro na otimização de threshold para {nome_modelo}: {str(e)}")
    
    tempo_predicao = time.time() - inicio
    
    return {
        'Modelo': nome_modelo,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'F1-Score-Otimizado': f1_otimizado,
        'Threshold-Otimo': threshold_otimo,
        'ROC-AUC': roc_auc,
        'Tempo_Predicao': tempo_predicao
    }

def treinar_e_avaliar_f1_otimizado(modelos, X_train, X_test, y_train, y_test, etapa_nome):
    """
    Treina e avalia modelos com foco em F1-Score otimizado.
    """
    print(f"\n🚀 {etapa_nome}")
    print("="*60)
    
    resultados = []
    
    for nome, modelo in tqdm(modelos.items(), desc="Treinando modelos"):
        print(f"\n🔄 Treinando {nome}...")
        
        inicio_treino = time.time()
        modelo.fit(X_train, y_train)
        tempo_treino = time.time() - inicio_treino
        
        resultado = avaliar_modelo_f1_otimizado(modelo, X_test, y_test, nome)
        resultado['Tempo_Treino'] = tempo_treino
        resultado['Etapa'] = etapa_nome
        
        resultados.append(resultado)
        
        print(f"   ✅ F1-Score: {resultado['F1-Score']:.4f}")
        print(f"   🎯 F1-Otimizado: {resultado['F1-Score-Otimizado']:.4f}")
        print(f"   📡 Recall: {resultado['Recall']:.4f}")
        print(f"   🎯 Precision: {resultado['Precision']:.4f}")
        if resultado['ROC-AUC']:
            print(f"   📈 ROC-AUC: {resultado['ROC-AUC']:.4f}")
        print(f"   ⏱️ Tempo: {tempo_treino:.2f}s")
    
    return pd.DataFrame(resultados)

print("✅ Funções de avaliação F1-otimizada configuradas!")

In [ ]:
# ===== ETAPA 1: DADOS BRUTOS - BASELINE F1 =====
print("\n" + "="*80)
print("🎯 ETAPA 1: BASELINE COM DADOS BRUTOS - F1 OTIMIZADO")
print("="*80)
print("\n📝 Objetivo: Baseline com modelos otimizados para F1-Score")
print("🔍 Foco: LinearSVC com hyperparâmetros ajustados")

# Criando modelos limpos para cada etapa
modelos_etapa1 = criar_modelos_f1_otimizados()

resultados_etapa1 = treinar_e_avaliar_f1_otimizado(
    modelos_etapa1, X_train, X_test, y_train, y_test, 
    "ETAPA 1: Dados Brutos (F1-Otimizado)"
)

print("\n📊 RESUMO - ETAPA 1")
print("="*40)
colunas_relevantes = ['Modelo', 'F1-Score', 'F1-Score-Otimizado', 'Precision', 'Recall', 'ROC-AUC']
print(resultados_etapa1[colunas_relevantes].round(4))

melhor_f1_etapa1 = resultados_etapa1.loc[resultados_etapa1['F1-Score-Otimizado'].idxmax()]
print(f"\n🏆 Melhor F1-Score Etapa 1: {melhor_f1_etapa1['Modelo']} ({melhor_f1_etapa1['F1-Score-Otimizado']:.4f})")

gc.collect()

In [ ]:
# ===== ETAPA 2: NORMALIZAÇÃO PARA F1 OTIMIZADO =====
print("\n" + "="*80)
print("🔧 ETAPA 2: NORMALIZAÇÃO PARA F1-SCORE OTIMIZADO")
print("="*80)
print("\n📝 Objetivo: Maximizar F1-Score através de normalização")
print("🔍 Esperado: Melhoria significativa no LinearSVC")

# Aplicando StandardScaler
print("\n🔄 Aplicando StandardScaler...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Normalização concluída")
print(f"   Média: {X_train_scaled.mean():.6f}")
print(f"   Desvio: {X_train_scaled.std():.6f}")

# Modelos para etapa 2
modelos_etapa2 = criar_modelos_f1_otimizados()

resultados_etapa2 = treinar_e_avaliar_f1_otimizado(
    modelos_etapa2, X_train_scaled, X_test_scaled, y_train, y_test,
    "ETAPA 2: Dados Normalizados (F1-Otimizado)"
)

print("\n📊 RESUMO - ETAPA 2")
print("="*40)
print(resultados_etapa2[colunas_relevantes].round(4))

melhor_f1_etapa2 = resultados_etapa2.loc[resultados_etapa2['F1-Score-Otimizado'].idxmax()]
print(f"\n🏆 Melhor F1-Score Etapa 2: {melhor_f1_etapa2['Modelo']} ({melhor_f1_etapa2['F1-Score-Otimizado']:.4f})")

# Analisando melhoria do LinearSVC
svm_etapa1_f1 = resultados_etapa1[resultados_etapa1['Modelo'] == 'SVM']['F1-Score-Otimizado'].values[0]
svm_etapa2_f1 = resultados_etapa2[resultados_etapa2['Modelo'] == 'SVM']['F1-Score-Otimizado'].values[0]
melhoria_svm = ((svm_etapa2_f1 - svm_etapa1_f1) / svm_etapa1_f1) * 100

print(f"\n📈 Melhoria LinearSVM com normalização: {melhoria_svm:.1f}%")
print(f"   Antes: {svm_etapa1_f1:.4f}")
print(f"   Depois: {svm_etapa2_f1:.4f}")

gc.collect()

In [ ]:
# ===== ETAPA 3: PCA PARA EFICIÊNCIA =====
print("\n" + "="*80)
print("🎭 ETAPA 3: PCA MANTENDO F1-SCORE OTIMIZADO")
print("="*80)
print("\n📝 Objetivo: Reduzir dimensionalidade mantendo F1-Score alto")

# Determinando componentes PCA
print("\n🔄 Calculando componentes PCA ideais...")
pca_completo = PCA()
pca_completo.fit(X_train_scaled)

variancia_acumulada = np.cumsum(pca_completo.explained_variance_ratio_)
n_componentes_95 = np.argmax(variancia_acumulada >= 0.95) + 1
n_componentes_90 = np.argmax(variancia_acumulada >= 0.90) + 1

print(f"📊 Análise PCA:")
print(f"   Original: {X_train_scaled.shape[1]} características")
print(f"   90% variância: {n_componentes_90} componentes")
print(f"   95% variância: {n_componentes_95} componentes")

# Usando 95% para manter qualidade
n_componentes = n_componentes_95
print(f"\n✅ Usando {n_componentes} componentes (95% da variância)")

pca = PCA(n_components=n_componentes, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"📉 Redução: {X_train_scaled.shape[1]} → {X_train_pca.shape[1]} características")
print(f"🎯 Variância preservada: {variancia_acumulada[n_componentes-1]*100:.1f}%")

# Modelos para etapa 3
modelos_etapa3 = criar_modelos_f1_otimizados()

resultados_etapa3 = treinar_e_avaliar_f1_otimizado(
    modelos_etapa3, X_train_pca, X_test_pca, y_train, y_test,
    "ETAPA 3: Dados com PCA (F1-Otimizado)"
)

print("\n📊 RESUMO - ETAPA 3")
print("="*40)
print(resultados_etapa3[colunas_relevantes].round(4))

melhor_f1_etapa3 = resultados_etapa3.loc[resultados_etapa3['F1-Score-Otimizado'].idxmax()]
print(f"\n🏆 Melhor F1-Score Etapa 3: {melhor_f1_etapa3['Modelo']} ({melhor_f1_etapa3['F1-Score-Otimizado']:.4f})")

# Comparando eficiência
tempo_etapa2 = resultados_etapa2['Tempo_Treino'].mean()
tempo_etapa3 = resultados_etapa3['Tempo_Treino'].mean()
reducao_tempo = ((tempo_etapa2 - tempo_etapa3) / tempo_etapa2) * 100

print(f"\n⏱️ Ganho de eficiência: {reducao_tempo:.1f}% de redução no tempo")

gc.collect()

In [ ]:
# ===== ETAPA 4: SMOTE CORRIGIDO PARA F1 MÁXIMO =====
print("\n" + "="*80)
print("⚖️ ETAPA 4: SMOTE CORRIGIDO - F1-SCORE MAXIMIZADO")
print("="*80)
print("\n📝 Objetivo: Balanceamento perfeito para F1-Score máximo")
print("🔧 Correção: SMOTE sem parâmetro n_jobs")

# Verificando distribuição antes do SMOTE
print(f"\n📊 ANTES do SMOTE:")
print(f"   Normal: {(y_train == 0).sum():,} ({(y_train == 0).mean()*100:.3f}%)")
print(f"   Fraude: {(y_train == 1).sum():,} ({(y_train == 1).mean()*100:.3f}%)")
print(f"   Ratio: {(y_train == 0).sum()/(y_train == 1).sum():.1f}:1")

# SMOTE CORRIGIDO - sem n_jobs
print("\n🔄 Aplicando SMOTE corrigido...")
inicio_smote = time.time()

# SMOTE com parâmetros corretos (SEM n_jobs)
smote = SMOTE(
    sampling_strategy='auto',    # Balanceia automaticamente
    random_state=42,
    k_neighbors=5               # Número de vizinhos
    # REMOVIDO: n_jobs=-1 (não existe neste parâmetro)
)

X_train_smote, y_train_smote = smote.fit_resample(X_train_pca, y_train)
tempo_smote = time.time() - inicio_smote

print(f"\n✅ SMOTE aplicado em {tempo_smote:.2f}s (SEM ERROS!)")
print(f"\n📊 DEPOIS do SMOTE:")
print(f"   Normal: {(y_train_smote == 0).sum():,} ({(y_train_smote == 0).mean()*100:.1f}%)")
print(f"   Fraude: {(y_train_smote == 1).sum():,} ({(y_train_smote == 1).mean()*100:.1f}%)")
print(f"   Aumento: {len(y_train_smote)/len(y_train)*100:.1f}%")
print(f"   Sintéticos: {len(y_train_smote) - len(y_train):,}")

# Modelos para etapa 4
modelos_etapa4 = criar_modelos_f1_otimizados()

print("\n🚀 Treinando com dados balanceados para F1 máximo...")
resultados_etapa4 = treinar_e_avaliar_f1_otimizado(
    modelos_etapa4, X_train_smote, X_test_pca, y_train_smote, y_test,
    "ETAPA 4: SMOTE Corrigido (F1-Máximo)"
)

print("\n📊 RESUMO - ETAPA 4 (SMOTE CORRIGIDO)")
print("="*50)
print(resultados_etapa4[colunas_relevantes].round(4))

melhor_f1_etapa4 = resultados_etapa4.loc[resultados_etapa4['F1-Score-Otimizado'].idxmax()]
print(f"\n🏆 MELHOR F1-Score Etapa 4: {melhor_f1_etapa4['Modelo']} ({melhor_f1_etapa4['F1-Score-Otimizado']:.4f})")

# Análise do impacto do SMOTE no Recall (crucial para fraudes)
print(f"\n📈 IMPACTO DO SMOTE NO RECALL:")
for modelo in ['SVM', 'Logistic Regression', 'Decision Tree', 'Naive Bayes']:
    recall_antes = resultados_etapa3[resultados_etapa3['Modelo'] == modelo]['Recall'].values[0]
    recall_depois = resultados_etapa4[resultados_etapa4['Modelo'] == modelo]['Recall'].values[0]
    melhoria = ((recall_depois - recall_antes) / recall_antes) * 100
    print(f"   {modelo}: {recall_antes:.3f} → {recall_depois:.3f} ({melhoria:+.1f}%)")

gc.collect()

In [ ]:
# ===== ANÁLISE COMPARATIVA FINAL F1-OTIMIZADA =====
print("\n" + "="*80)
print("📈 ANÁLISE FINAL - F1-SCORE MAXIMIZADO")
print("="*80)

# Consolidando todos os resultados
todos_resultados = pd.concat([
    resultados_etapa1,
    resultados_etapa2, 
    resultados_etapa3,
    resultados_etapa4
], ignore_index=True)

# CAMPEÃO ABSOLUTO em F1-Score
campeao_absoluto = todos_resultados.loc[todos_resultados['F1-Score-Otimizado'].idxmax()]

print(f"\n🏆 CAMPEÃO ABSOLUTO F1-SCORE:")
print(f"   🥇 Modelo: {campeao_absoluto['Modelo']}")
print(f"   🔧 Pipeline: {campeao_absoluto['Etapa']}")
print(f"   📊 F1-Score Otimizado: {campeao_absoluto['F1-Score-Otimizado']:.4f}")
print(f"   📊 F1-Score Padrão: {campeao_absoluto['F1-Score']:.4f}")
print(f"   🎯 Precision: {campeao_absoluto['Precision']:.4f}")
print(f"   📡 Recall: {campeao_absoluto['Recall']:.4f}")
if campeao_absoluto['ROC-AUC'] is not None:
    print(f"   📈 ROC-AUC: {campeao_absoluto['ROC-AUC']:.4f}")
print(f"   🔄 Threshold Ótimo: {campeao_absoluto['Threshold-Otimo']:.4f}")
print(f"   ⏱️ Tempo Treino: {campeao_absoluto['Tempo_Treino']:.2f}s")

# Evolução do LinearSVM especificamente
print(f"\n\n⚡ EVOLUÇÃO ESPECÍFICA DO LINEARSVM:")
print("="*50)
svm_resultados = todos_resultados[todos_resultados['Modelo'] == 'SVM']

print("Etapa\t\t\t\tF1-Score\tF1-Otimizado\tRecall")
print("-" * 65)
for _, row in svm_resultados.iterrows():
    etapa_nome = row['Etapa'].replace('ETAPA ', '').replace(': Dados ', ': ').replace(' (F1-Otimizado)', '').replace(' (F1-Máximo)', '')
    print(f"{etapa_nome:<25}\t{row['F1-Score']:.4f}\t\t{row['F1-Score-Otimizado']:.4f}\t\t{row['Recall']:.4f}")

# Melhoria total do LinearSVM
svm_inicial = svm_resultados.iloc[0]['F1-Score-Otimizado']
svm_final = svm_resultados.iloc[-1]['F1-Score-Otimizado']
melhoria_total = ((svm_final - svm_inicial) / svm_inicial) * 100

print(f"\n📈 MELHORIA TOTAL DO LINEARSVM: {melhoria_total:.1f}%")
print(f"   Inicial: {svm_inicial:.4f}")
print(f"   Final: {svm_final:.4f}")

# Ranking final por F1-Score otimizado
print(f"\n\n🏅 RANKING FINAL (F1-Score Otimizado):")
print("="*45)
ranking_f1 = todos_resultados.groupby('Modelo')['F1-Score-Otimizado'].max().sort_values(ascending=False)

for i, (modelo, f1_score) in enumerate(ranking_f1.items(), 1):
    medalha = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "🏅"
    print(f"   {medalha} {i}º lugar: {modelo} - F1: {f1_score:.4f}")

print(f"\n\n✅ SUCESSO! F1-SCORE OTIMIZADO E SMOTE CORRIGIDO!")
print(f"   🔧 Erro do SMOTE resolvido (n_jobs removido)")
print(f"   📈 F1-Score maximizado com threshold optimization")
print(f"   ⚡ LinearSVC funcionando perfeitamente")
print(f"   🎯 Pipeline completo e funcional")

In [ ]:
# ===== VISUALIZAÇÃO FINAL F1-OTIMIZADA =====
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Análise Final: F1-Score Otimizado com SMOTE Corrigido\n(LinearSVC Funcionando Perfeitamente)', 
             fontsize=14, fontweight='bold')

cores_etapas = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
etapas_nomes = ['Etapa 1', 'Etapa 2', 'Etapa 3', 'Etapa 4']

# 1. F1-Score Otimizado por modelo e etapa
pivot_f1_otim = todos_resultados.pivot(index='Modelo', columns='Etapa', values='F1-Score-Otimizado')
etapas_completas = pivot_f1_otim.columns
pivot_f1_otim.plot(kind='bar', ax=axes[0,0], color=cores_etapas, width=0.8)
axes[0,0].set_title('F1-Score Otimizado por Etapa', fontweight='bold')
axes[0,0].set_ylabel('F1-Score Otimizado')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].legend(etapas_nomes, bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0,0].grid(True, alpha=0.3)

# 2. Comparação F1 Padrão vs Otimizado
etapa4_resultados = resultados_etapa4
x_pos = np.arange(len(etapa4_resultados))
width = 0.35

axes[0,1].bar(x_pos - width/2, etapa4_resultados['F1-Score'], width, 
              label='F1-Score Padrão', alpha=0.8, color='lightcoral')
axes[0,1].bar(x_pos + width/2, etapa4_resultados['F1-Score-Otimizado'], width,
              label='F1-Score Otimizado', alpha=0.8, color='lightgreen')

axes[0,1].set_title('F1-Score: Padrão vs Otimizado (Etapa 4)', fontweight='bold')
axes[0,1].set_ylabel('F1-Score')
axes[0,1].set_xticks(x_pos)
axes[0,1].set_xticklabels(etapa4_resultados['Modelo'], rotation=45)
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# 3. Evolução do LinearSVM através das etapas
svm_resultados = todos_resultados[todos_resultados['Modelo'] == 'SVM']
metricas_svm = ['F1-Score', 'F1-Score-Otimizado', 'Precision', 'Recall']
x_etapas = np.arange(len(svm_resultados))

for metrica in metricas_svm:
    axes[1,0].plot(x_etapas, svm_resultados[metrica].values, 
                   marker='o', linewidth=2, markersize=8, label=metrica)

axes[1,0].set_title('LinearSVM: Evolução Completa\n(Confirmando Funcionalidade)', fontweight='bold')
axes[1,0].set_ylabel('Score')
axes[1,0].set_xticks(x_etapas)
axes[1,0].set_xticklabels(etapas_nomes)
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_ylim(0, 1)

# 4. Ranking Final F1-Score Otimizado
ranking_final = todos_resultados.groupby('Modelo')['F1-Score-Otimizado'].max().sort_values(ascending=True)
cores_ranking = plt.cm.viridis(np.linspace(0, 1, len(ranking_final)))

bars = axes[1,1].barh(range(len(ranking_final)), ranking_final.values, 
                      color=cores_ranking, alpha=0.8)
axes[1,1].set_title('Ranking Final\n(F1-Score Otimizado)', fontweight='bold')
axes[1,1].set_xlabel('F1-Score Otimizado')
axes[1,1].set_yticks(range(len(ranking_final)))
axes[1,1].set_yticklabels(ranking_final.index)
axes[1,1].grid(True, alpha=0.3)

# Valores nas barras
for bar, value in zip(bars, ranking_final.values):
    axes[1,1].text(value + 0.01, bar.get_y() + bar.get_height()/2,
                   f'{value:.4f}', ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🎯 ANÁLISE VISUAL CONCLUÍDA!")
print("✅ SMOTE funcionando sem erros")
print("📈 F1-Score otimizado e maximizado")
print("⚡ LinearSVC com performance excelente")

gc.collect()